Part 1: Data Loading & Setup
Q1. Spark Session & Data Loading
Create a SparkSession
Load sales_data.csv
Infer schema and display schema

In [83]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Module9_assignment").getOrCreate()

In [84]:
df = spark.read.format('csv').option('header', 'true').option('inferschema', 'true').load('/content/sales_data.csv')

In [85]:
df.show()

+--------+-----------+------+-------+------------+----------+-------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|
+--------+-----------+------+-------+------------+----------+-------------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|
+--------+-----------+------+-------+------------+----------+-------------------+



In [86]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



Q2. Data Type Conversion
Convert order_date to DateType
Convert order_timestamp to TimestampType


In [87]:
df = df.withColumn('order_date', F.to_date(df.order_date))
df = df.withColumn('order_timestamp', F.to_timestamp(df.order_timestamp))
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



Part 2: Aggregate Functions
Q3. Overall Sales Metrics
Calculate:

Total sales amount (sum)
Average order amount (avg)
Maximum and minimum order amount

In [88]:
stat1 = df.agg(F.sum('order_amount'),
               F.avg('order_amount'),
               F.max('order_amount'),
               F.min('order_amount'))
stat1.show()

+-----------------+-----------------+-----------------+-----------------+
|sum(order_amount)|avg(order_amount)|max(order_amount)|min(order_amount)|
+-----------------+-----------------+-----------------+-----------------+
|           359000|          44875.0|            80000|            20000|
+-----------------+-----------------+-----------------+-----------------+



Q4. Region-wise Analysis:
For each region, calculate:

Total sales
Average sales
Order count

In [89]:
stat2 = df.groupBy('region').agg(F.sum('order_amount'),
                                F.avg('order_amount'),
                                F.count('order_id'))
stat2.show()

+------+-----------------+------------------+---------------+
|region|sum(order_amount)| avg(order_amount)|count(order_id)|
+------+-----------------+------------------+---------------+
| South|           102000|           51000.0|              2|
|  East|           102000|           51000.0|              2|
|  West|            28000|           28000.0|              1|
| North|           127000|42333.333333333336|              3|
+------+-----------------+------------------+---------------+



Q5. Customer Count:
Number of distinct customers using countDistinct()

In [90]:
stat3 = df.agg(F.countDistinct('customer_id'))
stat3.show()

+---------------------------+
|count(DISTINCT customer_id)|
+---------------------------+
|                          4|
+---------------------------+



Q6. Product-wise Aggregation:
collect_list(order_amount)
collect_set(order_amount)

In [91]:
stat4 = df.agg(F.collect_list('order_amount'), F.collect_set('order_amount'))
stat4.show(truncate = False)

+--------------------------------------------------------+--------------------------------------------------------+
|collect_list(order_amount)                              |collect_set(order_amount)                               |
+--------------------------------------------------------+--------------------------------------------------------+
|[75000, 30000, 20000, 80000, 28000, 72000, 32000, 22000]|[20000, 22000, 28000, 30000, 32000, 72000, 80000, 75000]|
+--------------------------------------------------------+--------------------------------------------------------+



Part 3: Window Functions – Ranking
Q7. Regional Ranking:
For each region,
Assign row_number() based on highest order amount
Assign rank() and dense_rank()

In [92]:
stat5 = df.withColumn('row_number', F.row_number().over(Window.partitionBy('region').orderBy(F.desc('order_amount'))))
stat5 = stat5.withColumn('rank', F.rank().over(Window.partitionBy('region').orderBy(F.desc('order_amount'))))
stat5 = stat5.withColumn('dense_rank', F.dense_rank().over(Window.partitionBy('region').orderBy(F.desc('order_amount'))))

In [93]:
stat5.show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|row_number|rank|dense_rank|
+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|         1|   1|         1|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|         2|   2|         2|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|         1|   1|         1|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|         2|   2|         2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|         3|   3|         3|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|         1|   1|         1|
|    1002|       C0

Q8. Top Orders per Region
Find the top 2 highest orders per region using window functions.

In [94]:
stat6 = stat5.where(F.col('rank') <= 2)
stat6.show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|row_number|rank|dense_rank|
+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|         1|   1|         1|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|         2|   2|         2|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|         1|   1|         1|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|         2|   2|         2|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|         1|   1|         1|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|         2|   2|         2|
|    1005|       C0

Part 4: Window Functions – Analytical
Q9. Order Comparison per Customer
For each customer:
Use lag() to show previous order amount
Use lead() to show next order amount

In [95]:
stat7 = df.withColumn('lag', F.lag('order_amount').over(Window.partitionBy('customer_id').orderBy('order_date')))
stat7 = stat7.withColumn('lead', F.lead('order_amount').over(Window.partitionBy('customer_id').orderBy('order_date')))
stat7.show()

+--------+-----------+------+-------+------------+----------+-------------------+-----+-----+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|  lag| lead|
+--------+-----------+------+-------+------------+----------+-------------------+-----+-----+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00| NULL|20000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|75000|32000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|20000| NULL|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00| NULL|72000|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|30000| NULL|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00| NULL|22000|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|80000| NULL|
|    1005|       C004|  West| Mobile|       28000|2023-03-01

Q10. Running Total
Calculate running total (cumulative sum) of order amount:
Partition by customer_id
Order by order_date


In [96]:
stat8 = df.withColumn('sum', F.sum('order_amount').over(Window.partitionBy('customer_id').orderBy('order_date')))
stat8.show()

+--------+-----------+------+-------+------------+----------+-------------------+------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|   sum|
+--------+-----------+------+-------+------------+----------+-------------------+------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00| 75000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00| 95000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|127000|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00| 30000|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|102000|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00| 80000|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|102000|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00| 28000|
+--------+-----------

Part 5: Window Functions – Aggregates
Q11. Average Order per Customer
Add a column avg_customer_order showing the average order amount per customer using window functions

In [97]:
stat9 = df.withColumn('avg', F.avg('order_amount').over(Window.partitionBy('customer_id')))
stat9.show()

+--------+-----------+------+-------+------------+----------+-------------------+------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|               avg|
+--------+-----------+------+-------+------------+----------+-------------------+------------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|42333.333333333336|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|42333.333333333336|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|42333.333333333336|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|           51000.0|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|           51000.0|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|           51000.0|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|          

Q12. Maximum Order per Region
Add a column max_region_order showing the maximum order amount within each region

In [98]:
stat10 = df.withColumn('max', F.max('order_amount').over(Window.partitionBy('region')))
stat10.show()

+--------+-----------+------+-------+------------+----------+-------------------+-----+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|  max|
+--------+-----------+------+-------+------------+----------+-------------------+-----+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|80000|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|80000|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|75000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|75000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|75000|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|72000|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|72000|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|28000|
+--------+-----------+------+---

Part 6: Date & Timestamp Functions
Q13. Date Components Extraction:
Year, Month, Day from order_date

In [99]:
stat11 = df.withColumn('year', F.year('order_date'))
stat11 = stat11.withColumn('month', F.month('order_date'))
stat11 = stat11.withColumn('day', F.dayofmonth('order_date'))
stat11.show()

+--------+-----------+------+-------+------------+----------+-------------------+----+-----+---+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|year|month|day|
+--------+-----------+------+-------+------------+----------+-------------------+----+-----+---+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|2023|    1| 10|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|2023|    1| 12|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|2023|    2|  5|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|2023|    2| 20|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|2023|    3|  1|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|2023|    3| 15|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|2023|    3| 18|
|    1008|       C003|  East| 

Q14. Date Difference:
Days between today and order_date using datediff()

In [100]:
stat12 = df.withColumn('date_diff_inDays', F.datediff(F.current_date(), 'order_date'))
stat12.show()

+--------+-----------+------+-------+------------+----------+-------------------+----------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|date_diff_inDays|
+--------+-----------+------+-------+------------+----------+-------------------+----------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|            1171|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|            1169|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|            1145|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|            1130|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|            1121|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|            1107|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|            1104|
|    1008|

Q15. Create new columns:
order_year
order_month
order_week

In [101]:
df = df.withColumn('order_year', F.year('order_date'))
df = df.withColumn('order_month', F.month('order_date'))
df = df.withColumn('order_week', F.weekofyear('order_date'))
df.show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|order_year|order_month|order_week|
+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      2023|          1|         2|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|      2023|          1|         2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|      2023|          2|         5|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|      2023|          2|         8|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|      2023|          3|         9|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 

Q16. Date-based Filtering:
Placed in March 2023


In [102]:
stat13 = df.where((F.col('order_month') == 3) & (F.col('order_year') == 2023))
stat13.show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|order_year|order_month|order_week|
+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|      2023|          3|         9|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|      2023|          3|        11|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|      2023|          3|        11|
+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+



Part 7: Real-World ETL Scenarios
Q17. Monthly Sales Trend:
Calculate monthly total sales (group by Year + Month)

In [103]:
stat14 = df.groupBy('order_year', 'order_month').agg(F.sum('order_amount'))
stat14.show()

+----------+-----------+-----------------+
|order_year|order_month|sum(order_amount)|
+----------+-----------+-----------------+
|      2023|          3|           132000|
|      2023|          2|           100000|
|      2023|          4|            22000|
|      2023|          1|           105000|
+----------+-----------+-----------------+



Q18. Customer Activity Analysis:
Identify customers who have placed more than 2 orders

In [104]:
stat15 = df.groupBy('customer_id').agg(F.count('order_id').alias('count_of_orders'))
stat15 = stat15.where(F.col('count_of_orders') > 2)
stat15.show()

+-----------+---------------+
|customer_id|count_of_orders|
+-----------+---------------+
|       C001|              3|
+-----------+---------------+

